# 🔄 Warmstart — Stand Mo 24.08.2026 (Tag 37) · weiter am Mo 07.09.

## Wo ich stehe: A9 bei 7 von 9
✅ mittelwerte · zentriere · kovarianzmatrix · eigenwerte_2x2 · eigenvektor_2x2 · scores · erklaerte_varianz
⬜ pca(daten) · standardisiere(daten) — dazu Break-it und NumPy-Reveal

## Der erste Griff: pca(daten)
Enthält KEINE neue Rechnung. Ruft nur der Reihe nach auf, was schon läuft:
zentriere → kovarianzmatrix → eigenwerte_2x2 → eigenvektor_2x2 (ZWEIMAL, einmal je Eigenwert)
→ scores → erklaerte_varianz
Rückgabe: ein dict mit C, Eigenwerten, Komponenten, Scores, Anteilen.
Die Aufrufzeilen stehen unten in der Zelle schon fertig — sie müssen nur hinein.

## Meine Kontrollen — die haben am 24.08. drei Fehler gefunden
| Was | Sollwert |
|---|---|
| Form der Scores | so viele Zeilen wie Daten, je so viele Zahlen wie Komponenten |
| Spaltenmittel der Scores | 0 |
| Spaltenvarianz der Scores (n−1) | = der zugehörige Eigenwert |
| Länge einer Score-Zeile | = Länge der zentrierten Zeile |
| v₁ · v₂ | 0 |
| Summe der Anteile | 100 |

## Mein Datensatz — nicht entartet, der echte Test
daten = [[2, 8], [8, 9], [8, 13]]
C = [[12, 6], [6, 7]] · λ = [16, 3] · PC1 = (0.8321, 0.5547)
scores  = [[-4.4376, -0.5547], [1.1094, 1.9415], [3.3282, -1.3868]]
anteile = [84.21, 15.79]

## Selbsttest 1 — Heft-Werte, aber ENTARTET (gleiche Varianzen), nur Zahlenprobe
daten = [[6, 11], [5, 9], [4, 10]]
C = [[1.0, 0.5], [0.5, 1.0]] · λ = [1.5, 0.5] · PC1 = (0.7071, 0.7071)
scores  = [[1.414, 0.0], [-0.707, 0.707], [-0.707, -0.707]]
anteile = [75.0, 25.0]

## Regel: Jede Bausitzung endet mit Kernel-Neustart und Durchlauf von oben.

In [1]:
# Aufgabe 9 PCA von Hand jetzt mit ungleichen Varianzen

daten = [[2, 8], [8, 9], [8, 13]] # Beide Datensätze angelegt, um zwischen ihnen switchen zu können
#daten = [[6, 11], [5, 9], [4, 10]] 

def mittelwerte(daten):
    spaltenmittel = []
    summe_1 = 0
    summe_2 = 0
    for kd_merkmale in daten:
        summe_1 += kd_merkmale[0]
        summe_2 += kd_merkmale[1]
    
    spaltenmittel.append(summe_1 / len(daten))
    spaltenmittel.append(summe_2 / len(daten))
    
    return spaltenmittel

def zentriere(daten):
    mittel = mittelwerte(daten)
    zentrierte_daten = []
    for kd_daten in daten:
        z_wert_1 = kd_daten[0] - mittel[0]
        z_wert_2 = kd_daten[1] - mittel[1]

        z_daten = []
        z_daten.append(z_wert_1)
        z_daten.append(z_wert_2)

        zentrierte_daten.append(z_daten)

    return zentrierte_daten

def spalte(daten, j):
    s_list = []
    for zeile in daten:
        s_list.append(zeile[j])
    return s_list

def kovarianzmatrix(X):
    C = []
    n = len(X)
    for i in range(len(X[0])):
        C_line = []
        sp_i = spalte(X, i) # eine Ebene höher, damit sie nicht jedes mal in der Schleife unten neu ermittelt wird obwohl sie auf der Ebene konstant bleibt.
        for j in range(len(X[0])):
            sp_j = spalte(X, j)
            
            summe = 0
            for k in range(n):
                summe += sp_i[k] * sp_j[k]

            covar = summe / (n-1)
            C_line.append(covar)
        C.append(C_line)

    return C

def eigenwerte_2x2(C):

    spur = C[0][0] + C[1][1]
    det = C[0][0] * C[1][1] - C[0][1] * C[1][0]

    sroot = ((spur**2 - 4*det)**0.5)

    lam_high = spur/2 + sroot/2
    lam_low = spur/2 - sroot/2

    return [lam_high, lam_low]

def eigenvektor_2x2(C, lam):
    v1 = C[0][1] # „Länge frei wählbar, hier so gesetzt, dass keine Division nötig ist — das Normieren macht die Wahl später gegenstandslos."
    v2 = lam - C[0][0]
    v_betrag = (v1**2 + v2**2)**0.5
    if v_betrag == 0:
        v1 = lam - C[1][1]
        v2 = C[1][0]
        v_betrag = (v1**2 + v2**2)**0.5
        if v_betrag == 0:
            v1 = 1
            v2 = 0
            v_betrag = 1 # Nicht mehr die Formel, da dieser Wert 1 sein muss bei gegebenem Vektor (1,0)

    return [v1 / v_betrag, v2 / v_betrag]

def scores(X, pcs):
    kd_scores_total = []
    for zeile in X:
        kd_scores = []
        for i in range(len(pcs)):
            kd_pcs = pcs[i]
            kd_score_pcs = 0
            for j in range(len(zeile)):
                kd_score_pcs += zeile[j] * kd_pcs[j]            
            kd_scores.append(kd_score_pcs)
        kd_scores_total.append(kd_scores)

    return kd_scores_total

def erklaerte_varianz(lams):
    gesamtvarianz = 0
    pc_beitraege = []
    for lam in lams:
        gesamtvarianz += lam
    for lam in lams:
        beitrag_n = lam / gesamtvarianz * 100
        pc_beitraege.append(beitrag_n)

    return pc_beitraege

X = zentriere(daten)
C = kovarianzmatrix(X)
lams = eigenwerte_2x2(C)
lam_high = eigenwerte_2x2(C)[0]
lam_low = eigenwerte_2x2(C)[1]
eigvek_high = eigenvektor_2x2(C, lam_high)
eigvek_low = eigenvektor_2x2(C, lam_low)
pcs = [eigvek_high, eigvek_low]
print(f'Mittelwerte: {mittelwerte(daten)}')
print(f'Zentrierte Matrix: {X}')
print(f'Spalten der zentr. Matrix: {spalte(X, 0), spalte(X, 1)}')
print(f'Kovarianzmatrix: {C}')
print(f'Eingenwerte: {eigenwerte_2x2(C)}')
print(f'Eigenvektor_high: {eigvek_high}')
print(f'Eigenvektor_low: {eigvek_low}')
print(f'Kundenscores high und low von K1, K2, K3: {scores(X, pcs)}')
print(f'Die jeweiligen PC-Beiträge absteigend: {erklaerte_varianz(lams)}')


Mittelwerte: [6.0, 10.0]
Zentrierte Matrix: [[-4.0, -2.0], [2.0, -1.0], [2.0, 3.0]]
Spalten der zentr. Matrix: ([-4.0, 2.0, 2.0], [-2.0, -1.0, 3.0])
Kovarianzmatrix: [[12.0, 6.0], [6.0, 7.0]]
Eingenwerte: [16.0, 3.0]
Eigenvektor_high: [0.8320502943378437, 0.5547001962252291]
Eigenvektor_low: [0.554700196225229, -0.8320502943378436]
Kundenscores high und low von K1, K2, K3: [[-4.437601569801833, -0.5547001962252289], [1.1094003924504583, 1.9414506867883017], [3.328201177351375, -1.3867504905630728]]
Die jeweiligen PC-Beiträge absteigend: [84.21052631578947, 15.789473684210526]
